In [1]:
# Step 1: Setup and Installation
!pip install -q peft transformers datasets
!pip install pyarrow==8.0.0  # Downgrade pyarrow to a known compatible version
from transformers import AutoModelForSequenceClassification, AutoTokenizer, default_data_collator, get_linear_schedule_with_warmup
from peft import get_peft_config, get_peft_model, PromptTuningInit, PromptTuningConfig, TaskType
import torch
from torch.nn.functional import pad
from datasets import load_dataset, Dataset
import pandas as pd
from torch.utils.data import DataLoader
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 24.4.1 requires pyarrow<15.0.0a0,>=14.0.1, but you have pyarrow 16.1.0 which is incompatible.
ibis-framework 8.0.0 requires pyarrow<16,>=2, but you have pyarrow 16.1.0 which is incompatible.
  Using cached pyarrow-8.0.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (29.4 MB)
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 16.1.0
    Uninstalling pyarrow-16.1.0:
      Successfully uninstalled pyarrow-16.1.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 24.4.1 requires pyarrow<15.0.0a0,>=14.0.1, but you have pyarrow 8.0.0 which is incompatible.
datasets 2.20.0 requires pyarrow>=15.0.0, but you have pyarrow 8.0.0 which is incompatibl

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:

# Step 2: Define Configuration
device = "cuda"
model_name_or_path = 'google/muril-base-cased'  # Changed to MuRIL model
tokenizer_name_or_path = 'google/muril-base-cased'  # Changed to MuRIL model
peft_config = PromptTuningConfig(
    task_type=TaskType.SEQ_CLS,  # Correct task type for sequence classification
    prompt_tuning_init=PromptTuningInit.TEXT,
    num_virtual_tokens=8,
    prompt_tuning_init_text="What is the sentiment of this text?",  # Changed to a more specific prompt
    tokenizer_name_or_path=tokenizer_name_or_path,
)
dataset_name = "twitter_complaints"
checkpoint_name = f"{dataset_name}_{model_name_or_path}_{peft_config.peft_type}_{peft_config.task_type}_v1.pt".replace("/", "_")
text_column = "Tweet text"
label_column = "text_label"
max_length = 128  # Increased max_length to accommodate longer texts
lr = 3e-4
num_epochs = 50
batch_size = 8

In [4]:
column_names = [label_column,"blank" ,text_column]
# Load the dataset without headers and assign the column names


#1% - 52, 2%- 104, 10% - 520
number = 520

dataset_train = pd.read_csv("/content/drive/MyDrive/train_dataset_Telugu.csv", header=None, names=column_names).head(number)

dataset_test = pd.read_csv("/content/drive/MyDrive/test_dataset_Telugu.csv", header=None, names=column_names)

dataset_train.drop("blank", axis=1, inplace=True)
dataset_train[label_column] = dataset_train[label_column].map({0: 0, 4: 1})  # Map labels to 0 and 1 for binary classification
print(dataset_train.head())

dataset_test.drop("blank", axis=1, inplace=True)
dataset_test[label_column] = dataset_test[label_column].map({0: 0, 4: 1})  # Map labels to 0 and 1 for binary classification
print(dataset_test.head())

   text_label                                         Tweet text
0           0  @AAJTAK @ABPNEWSHINDI అన్ని వ్యతిరేకత నామో 4 భ...
1           0  @స్వామి 39 మీరు ఎంపి మరియు విద్యావంతుడైన వ్యక్...
2           0  @Bdutt @sagarikaghose మీ పేద #బుర్హాన్వానీ ఒక ...
3           1  అస్సాం అసెంబ్లీ జిఎస్‌టిపై రాజ్యాంగ సవరణ బిల్ల...
4           0  RT @Mnomics_: GST రద్దు కర్నే మెయిన్ 5 సాల్ బి...
   text_label                                         Tweet text
0           0  @AAJTAK @ABPNEWSHINDI అన్ని వ్యతిరేకత నామో 4 భ...
1           0  @Timesnow u మదర్స్ ఫకర్స్..మేము మరణం తరువాత కొ...
2           1  @narendramodi గొప్ప అడుగులు వేయండి, GSTని మాత్...
3           1  #Toi #newsindia ప్రభుత్వం GST సేకరణను నిర్వహిం...
4           1  #URI 28, #బ్రౌన్ 13 ఇక్కడ కింగ్‌స్టన్‌లో.2010 ...


In [5]:
# Now you have train_dataset and test_dataset for training and testing respectively
from datasets import Dataset
dataset_train = Dataset.from_pandas(dataset_train)
dataset_test = Dataset.from_pandas(dataset_test)

In [6]:
# Step 4: Tokenizer Initialization
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [8]:
# Step 5: Preprocess Function
def preprocess_function(examples):
    model_inputs = tokenizer(examples[text_column], truncation=True, max_length=max_length, padding="max_length")
    model_inputs["labels"] = examples[label_column]
    return model_inputs

In [9]:
# Step 6: Apply Preprocessing
processed_datasets_train = dataset_train.map(
    preprocess_function,
    batched=True,
    num_proc=1,
    remove_columns=dataset_train.column_names,
    load_from_cache_file=False,
    desc="Running tokenizer on dataset",
)
processed_datasets_test = dataset_test.map(
    preprocess_function,
    batched=True,
    num_proc=1,
    remove_columns=dataset_test.column_names,
    load_from_cache_file=False,
    desc="Running tokenizer on dataset",
)

Running tokenizer on dataset:   0%|          | 0/520 [00:00<?, ? examples/s]

Running tokenizer on dataset:   0%|          | 0/20882 [00:00<?, ? examples/s]

In [10]:
# Step 7: DataLoaders
train_dataloader = DataLoader(
    processed_datasets_train, shuffle=True, collate_fn=default_data_collator, batch_size=batch_size, pin_memory=True
)
test_dataloader = DataLoader(
    processed_datasets_test, shuffle=False, collate_fn=default_data_collator, batch_size=batch_size, pin_memory=True
)

In [11]:
# Step 8: Model Initialization
model = AutoModelForSequenceClassification.from_pretrained(model_name_or_path, num_labels=2)
model = get_peft_model(model, peft_config)
print(model.print_trainable_parameters())
# Add dropout to prevent overfitting
model.dropout = torch.nn.Dropout(0.1)

pytorch_model.bin:  76%|#######5  | 724M/953M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google/muril-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 7,682 || all params: 237,565,444 || trainable%: 0.0032
None


In [12]:
# Step 9: Optimizer and Scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
lr_scheduler = get_linear_schedule_with_warmup(optimizer=optimizer, num_warmup_steps=0, num_training_steps=(len(train_dataloader) * num_epochs))

In [14]:
# # Step 10: Training Loop
# model = model.to(device)
# for epoch in range(num_epochs):
#     model.train()
#     total_loss = 0
#     for step, batch in enumerate(tqdm(train_dataloader)):
#         batch = {k: v.to(device) for k, v in batch.items()}
#         outputs = model(**batch)
#         loss = outputs.loss
#         total_loss += loss.detach().float()
#         loss.backward()
#         optimizer.step()
#         lr_scheduler.step()
#         optimizer.zero_grad()

# Step 10: Training Loop
model = model.to(device)
accumulation_steps = 4
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    for step, batch in enumerate(tqdm(train_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        total_loss += loss.detach().float()
        loss = loss / accumulation_steps
        loss.backward()

        if (step + 1) % accumulation_steps == 0:
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()

        # Early stopping
        if eval_epoch_loss < best_eval_loss:
           best_eval_loss = eval_epoch_loss
           patience_counter = 0
           # Save the best model
           torch.save(model.state_dict(), checkpoint_name)
        else:
           patience_counter += 1
           if patience_counter >= patience:
               print("Early stopping")
               break

# Load the best model
model.load_state_dict(torch.load(checkpoint_name))
model.eval()

    train_epoch_loss = total_loss / len(train_dataloader)
    print(f"Epoch {epoch}: train_loss={train_epoch_loss}")#, eval_loss={eval_epoch_loss}

IndentationError: unexpected indent (<ipython-input-14-e2cd841a228d>, line 52)

In [15]:

#Training Loop
model = model.to(device)
accumulation_steps = 4
best_eval_loss = float('inf')
patience = 10  # Early stopping patience
patience_counter = 0

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    for step, batch in enumerate(tqdm(train_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        total_loss += loss.detach().float()
        loss = loss / accumulation_steps
        loss.backward()

        if (step + 1) % accumulation_steps == 0:
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()

    print(f"Epoch {epoch} Train Loss: {total_loss / len(train_dataloader)}")




    # Early stopping
    if eval_epoch_loss < best_eval_loss:
        best_eval_loss = eval_epoch_loss
        patience_counter = 0
        # Save the best model
        torch.save(model.state_dict(), checkpoint_name)
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping")
            break

# Load the best model
model.load_state_dict(torch.load(checkpoint_name))
model.eval()

train_epoch_loss = total_loss / len(train_dataloader)
    print(f"Epoch {epoch}: train_loss={train_epoch_loss}")#, eval_loss={eval_epoch_loss}

IndentationError: unexpected indent (<ipython-input-15-faa49cddb5b9>, line 47)

In [ ]:

    model.eval()
    eval_loss = 0
    eval_preds = []
    true_labels = []
    for step, batch in enumerate(tqdm(test_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)
        loss = outputs.loss
        eval_loss += loss.detach().float()
        eval_preds.extend(torch.argmax(outputs.logits, -1).detach().cpu().numpy())
        true_labels.extend(batch["labels"].detach().cpu().numpy())

    eval_epoch_loss = eval_loss / len(test_dataloader)
    train_epoch_loss = total_loss / len(train_dataloader)
    accuracy = accuracy_score(true_labels, eval_preds)
    f1 = f1_score(true_labels, eval_preds, average='weighted')
    recall = recall_score(true_labels, eval_preds, average='weighted')
    precision = precision_score(true_labels, eval_preds, average='weighted')

    print(f"Epoch {epoch}: train_loss={train_epoch_loss}, eval_loss={eval_epoch_loss}, accuracy={accuracy}, f1_score={f1}, recall={recall}, precision={precision}")

100%|██████████| 2611/2611 [02:55<00:00, 14.87it/s]


Epoch 49: train_loss=0.693512499332428, eval_loss=0.6930463910102844, accuracy=0.5090029690642659, f1_score=0.34338437740370553, recall=0.5090029690642659, precision=0.259084022516238


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [ ]:
accuracy = accuracy_score(true_labels, eval_preds)
f1 = f1_score(true_labels, eval_preds, average='weighted')
recall = recall_score(true_labels, eval_preds, average='weighted')
precision = precision_score(true_labels, eval_preds, average='weighted')

print(f"Accuracy: {accuracy}")
print(f"F1 Score: {f1}")
print(f"Recall: {recall}")
print(f"Precision: {precision}")

Accuracy: 0.5090029690642659
F1 Score: 0.34338437740370553
Recall: 0.5090029690642659
Precision: 0.259084022516238


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [ ]:
# After training, output the predictions
# Convert predictions and true labels to a DataFrame for easy analysis
results_df = pd.DataFrame({
    'true_label': true_labels,
    'predicted_label': eval_preds
})
print(results_df)

In [ ]:
# Save the results to a CSV file
results_df.to_csv("/content/drive/MyDrive/model_predictions.csv", index=False)
print("Predictions saved to /content/drive/MyDrive/model_predictions.csv")

# Or print the first few rows of the results
print(results_df.head())

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score